# Android App Store Market Analysis

Where should you launch an app? This analysis of 8,199 Google Play Store apps builds a three-part market intelligence framework: an Opportunity Score that ranks every category by demand relative to competition, a paid viability analysis that identifies the 11 categories where charging actually works, and a K-Means segmentation that maps each individual app to one of four structurally distinct market positions. The goal is a data-driven answer to a question every developer faces before writing a single line of code.

## Dataset

**Source:** Google Play Store data scraped by Lavanya Gupta in 2018 ([Kaggle](https://www.kaggle.com/lava18/google-play-store-apps)). 10,841 rows × 12 columns covering app name, category, rating, review count, size (MBs), install count, type (free/paid), price, content rating, and genres. Two columns (`Last_Updated`, `Android_Ver`) are dropped before analysis.

## Setup

In [1]:
import pandas as pd


## Display Configuration

In [2]:
# Show numeric output in decimal format e.g., 2.15
pd.options.display.float_format = '{:,.2f}'.format

## Load Data

In [3]:
df_apps = pd.read_csv('../../data/apps.csv')

## Data Cleaning

### Initial Inspection

The raw dataset has 10,841 rows across 12 columns. A random sample confirms mixed types: install counts are stored as formatted strings (e.g. `"10,000+"`), prices as dollar strings (e.g. `"$2.99"`). Both require conversion before any numeric analysis.

In [4]:
df_apps.shape

(10841, 12)

In [5]:
df_apps.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10841 entries, 0 to 10840
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   App             10841 non-null  object 
 1   Category        10841 non-null  object 
 2   Rating          9367 non-null   float64
 3   Reviews         10841 non-null  int64  
 4   Size_MBs        10841 non-null  float64
 5   Installs        10841 non-null  object 
 6   Type            10840 non-null  object 
 7   Price           10841 non-null  object 
 8   Content_Rating  10841 non-null  object 
 9   Genres          10841 non-null  object 
 10  Last_Updated    10841 non-null  object 
 11  Android_Ver     10839 non-null  object 
dtypes: float64(2), int64(1), object(9)
memory usage: 1016.5+ KB


In [6]:
df_apps.sample(5)

,App,Category,Rating,Reviews,Size_MBs,Installs,Type,Price,Content_Rating,Genres,Last_Updated,Android_Ver
2193,I am extremely Rich,LIFESTYLE,2.90,41,2.90,"1,000",Paid,$379.99,Everyone,Lifestyle,"July 1, 2018",4.0 and up
8484,Adobe Photoshop Fix,PHOTOGRAPHY,4.20,28390,38.00,"5,000,000",Free,0,Everyone,Photography,"July 19, 2017",5.0 and up
10666,Garena Free Fire,GAME,4.50,5534114,53.00,"100,000,000",Free,0,Teen,Action,"August 3, 2018",4.0.3 and up
2062,"Talkie Pro - Wi-Fi Calling, Chats, File Sharing",COMMUNICATION,4.50,201,3.50,"1,000",Paid,$2.99,Everyone,Communication,"January 6, 2018",Varies with device
5376,I will return his eggs,LIBRARIES_AND_DEMO,NaN,91,18.00,"100,000",Free,0,Everyone,Libraries & Demo,"November 16, 2017",2.3 and up


In [7]:
df_apps.head()

,App,Category,Rating,Reviews,Size_MBs,Installs,Type,Price,Content_Rating,Genres,Last_Updated,Android_Ver
0,Ak Parti Yardım Toplama,SOCIAL,NaN,0,8.70,0,Paid,$13.99,Teen,Social,"July 28, 2017",4.1 and up
1,Ain Arabic Kids Alif Ba ta,FAMILY,NaN,0,33.00,0,Paid,$2.99,Everyone,Education,"April 15, 2016",3.0 and up
2,Popsicle Launcher for Android P 9.0 launcher,PERSONALIZATION,NaN,0,5.50,0,Paid,$1.49,Everyone,Personalization,"July 11, 2018",4.2 and up
3,Command & Conquer: Rivals,FAMILY,NaN,0,19.00,0,NaN,0,Everyone 10+,Strategy,"June 28, 2018",Varies with device
4,CX Network,BUSINESS,NaN,0,10.00,0,Free,0,Everyone,Business,"August 6, 2018",4.1 and up


### Drop Unused Columns

`Last_Updated` and `Android_Ver` carry no signal for market analysis and are removed.

In [8]:
df_apps.drop(['Last_Updated', 'Android_Ver'], axis=1, inplace=True)

In [9]:
df_apps.head()

,App,Category,Rating,Reviews,Size_MBs,Installs,Type,Price,Content_Rating,Genres
0,Ak Parti Yardım Toplama,SOCIAL,NaN,0,8.70,0,Paid,$13.99,Teen,Social
1,Ain Arabic Kids Alif Ba ta,FAMILY,NaN,0,33.00,0,Paid,$2.99,Everyone,Education
2,Popsicle Launcher for Android P 9.0 launcher,PERSONALIZATION,NaN,0,5.50,0,Paid,$1.49,Everyone,Personalization
3,Command & Conquer: Rivals,FAMILY,NaN,0,19.00,0,NaN,0,Everyone 10+,Strategy
4,CX Network,BUSINESS,NaN,0,10.00,0,Free,0,Everyone,Business


### Remove NaN Ratings

Apps with no rating have never been rated — they represent pre-launch or zero-traction entries that would skew any rating-based analysis. Dropping NaN rows in the Rating column reduces the dataset from 10,841 to 9,367 rows.

In [10]:
df_apps.shape

(10841, 10)

In [11]:
nan_rows = df_apps[df_apps.Rating.isna()]
print(nan_rows.shape)
nan_rows.head()

(1474, 10)


,App,Category,Rating,Reviews,Size_MBs,Installs,Type,Price,Content_Rating,Genres
0,Ak Parti Yardım Toplama,SOCIAL,NaN,0,8.70,0,Paid,$13.99,Teen,Social
1,Ain Arabic Kids Alif Ba ta,FAMILY,NaN,0,33.00,0,Paid,$2.99,Everyone,Education
2,Popsicle Launcher for Android P 9.0 launcher,PERSONALIZATION,NaN,0,5.50,0,Paid,$1.49,Everyone,Personalization
3,Command & Conquer: Rivals,FAMILY,NaN,0,19.00,0,NaN,0,Everyone 10+,Strategy
4,CX Network,BUSINESS,NaN,0,10.00,0,Free,0,Everyone,Business


In [12]:
df_apps_clean = df_apps.dropna()
df_apps_clean.shape

(9367, 10)

### Remove Duplicates

Deduplication on `[App, Type, Price]` removes entries where the same app appears multiple times with identical pricing and type — common with scraped store data. Instagram, for example, appears multiple times. After deduplication: **8,199 apps**. This is the cleaned working dataset, `df_apps_clean`, used throughout all subsequent analysis.

In [13]:
df_apps_clean.shape

(9367, 10)

In [14]:
duplicated_rows = df_apps_clean[df_apps_clean.duplicated()]
duplicated_rows.shape
duplicated_rows.head()

,App,Category,Rating,Reviews,Size_MBs,Installs,Type,Price,Content_Rating,Genres
946,420 BZ Budeze Delivery,MEDICAL,5.00,2,11.00,100,Free,0,Mature 17+,Medical
1133,MouseMingle,DATING,2.70,3,3.90,100,Free,0,Mature 17+,Dating
1196,"Cardiac diagnosis (heart rate, arrhythmia)",MEDICAL,4.40,8,6.50,100,Paid,$12.99,Everyone,Medical
1231,Sway Medical,MEDICAL,5.00,3,22.00,100,Free,0,Everyone,Medical
1247,Chat Kids - Chat Room For Kids,DATING,4.70,6,4.90,100,Free,0,Mature 17+,Dating


In [15]:
# bad attempt at dropping duplicates

df_apps_clean = df_apps_clean.drop_duplicates()
df_apps_clean[df_apps_clean.App == "Instagram"]

,App,Category,Rating,Reviews,Size_MBs,Installs,Type,Price,Content_Rating,Genres
10806,Instagram,SOCIAL,4.50,66577313,5.30,"1,000,000,000",Free,0,Teen,Social
10808,Instagram,SOCIAL,4.50,66577446,5.30,"1,000,000,000",Free,0,Teen,Social
10810,Instagram,SOCIAL,4.50,66509917,5.30,"1,000,000,000",Free,0,Teen,Social


In [16]:
# need to specify the subset for identifying duplicates
df_apps_clean = df_apps_clean.drop_duplicates(subset=["App", "Type", "Price"])
df_apps_clean[df_apps_clean.App =="Instagram"]

,App,Category,Rating,Reviews,Size_MBs,Installs,Type,Price,Content_Rating,Genres
10806,Instagram,SOCIAL,4.50,66577313,5.30,"1,000,000,000",Free,0,Teen,Social


In [17]:
df_apps_clean.shape

(8199, 10)

In [18]:
df_apps.shape

(10841, 10)

## Preliminary Exploration

### Highest Rated Apps

Sorting by rating alone surfaces a known problem with star ratings: apps with very few reviews can achieve a perfect 5.0. Rating is a signal of quality only when paired with sufficient review volume — a constraint that becomes central to the segmentation analysis later.

In [19]:
df_apps.sort_values("Rating", ascending=False).head(10)

,App,Category,Rating,Reviews,Size_MBs,Installs,Type,Price,Content_Rating,Genres
21,KBA-EZ Health Guide,MEDICAL,5.00,4,25.00,1,Free,0,Everyone,Medical
1639,DC-014,PHOTOGRAPHY,5.00,3,16.00,500,Free,0,Everyone,Photography
1675,WPBS-DT,FAMILY,5.00,3,6.30,500,Free,0,Everyone,Entertainment
1677,CG FM,FAMILY,5.00,2,6.60,500,Free,0,Everyone,Entertainment
958,Food-Aw - Order Food Online in Aruba,FOOD_AND_DRINK,5.00,1,24.00,100,Free,0,Everyone,Food & Drink
957,CP Installer App,BUSINESS,5.00,4,24.00,100,Free,0,Everyone,Business
954,CJ'S TIRE AND AUTO INC.,PRODUCTIVITY,5.00,5,11.00,100,Free,0,Everyone,Productivity
1685,EK Bailey Preaching Conference,EVENTS,5.00,3,30.00,500,Free,0,Everyone,Events
951,BK Formula Calculator,TOOLS,5.00,6,11.00,100,Free,0,Everyone,Tools
1690,Exam Result BD,FAMILY,5.00,2,2.60,500,Free,0,Everyone,Education


### Largest Apps by Size

The largest apps cluster around 100 MBs, consistent with historical Play Store download size limits. Size functions as a proxy for product complexity — a feature included in the K-Means segmentation.

In [20]:
df_apps.sort_values("Size_MBs", ascending=False).head(10)

,App,Category,Rating,Reviews,Size_MBs,Installs,Type,Price,Content_Rating,Genres
8719,Draft Simulator for FUT 18,SPORTS,4.60,162933,100.00,"5,000,000",Free,0,Everyone,Sports
9943,Miami crime simulator,GAME,4.00,254518,100.00,"10,000,000",Free,0,Mature 17+,Action
10687,Hungry Shark Evolution,GAME,4.50,6074334,100.00,"100,000,000",Free,0,Teen,Arcade
8718,Mini Golf King - Multiplayer Game,GAME,4.50,531458,100.00,"5,000,000",Free,0,Everyone,Sports
4176,Car Crash III Beam DH Real Damage Simulator 2018,GAME,3.60,151,100.00,"10,000",Free,0,Everyone,Racing
10689,Hungry Shark Evolution,GAME,4.50,6071542,100.00,"100,000,000",Free,0,Teen,Arcade
7926,Post Bank,FINANCE,4.50,60449,100.00,"1,000,000",Free,0,Everyone,Finance
7927,The Walking Dead: Our World,GAME,4.00,22435,100.00,"1,000,000",Free,0,Teen,Action
7928,Stickman Legends: Shadow Wars,GAME,4.40,38419,100.00,"1,000,000",Paid,$0.99,Everyone 10+,Action
3144,Vi Trainer,HEALTH_AND_FITNESS,3.60,124,100.00,"5,000",Free,0,Everyone,Health & Fitness


### Most Reviewed Apps

Review count is dominated by a small number of global platforms (Facebook, WhatsApp, Instagram). The top 50 most-reviewed apps are almost entirely free, establishing an early signal that scale and paid pricing do not co-exist in most categories.

In [21]:
df_apps_clean.sort_values("Reviews", ascending=False).head()

,App,Category,Rating,Reviews,Size_MBs,Installs,Type,Price,Content_Rating,Genres
10805,Facebook,SOCIAL,4.10,78158306,5.30,"1,000,000,000",Free,0,Teen,Social
10785,WhatsApp Messenger,COMMUNICATION,4.40,69119316,3.50,"1,000,000,000",Free,0,Everyone,Communication
10806,Instagram,SOCIAL,4.50,66577313,5.30,"1,000,000,000",Free,0,Teen,Social
10784,Messenger – Text and Video Chat for Free,COMMUNICATION,4.00,56642847,3.50,"1,000,000,000",Free,0,Everyone,Communication
10650,Clash of Clans,GAME,4.60,44891723,98.00,"100,000,000",Free,0,Everyone 10+,Strategy


## Content Rating Distribution

Content rating classifies the intended audience age group. Understanding the distribution reveals who the Play Store is primarily built for — and which audience segments are underserved.

In [22]:
ratings = df_apps_clean.Content_Rating.value_counts()
ratings

Content_Rating
Everyone           6621
Teen                912
Mature 17+          357
Everyone 10+        305
Adults only 18+       3
Unrated               1
Name: count, dtype: int64

In [23]:
import pandas as pd
import plotly.express as px

In [24]:
fig = px.pie(labels=ratings.index, values=ratings.values)


fig.write_image('../../plots/content_rating_donut.png', scale=2)
fig.show()

In [25]:
# customizing chart
ratings.head()

Content_Rating
Everyone           6621
Teen                912
Mature 17+          357
Everyone 10+        305
Adults only 18+       3
Name: count, dtype: int64

In [26]:
fig = px.pie(labels=ratings.index,
values=ratings.values,
title="Content Rating",
names=ratings.index,
)
fig.update_traces(textposition='outside', textinfo='percent+label')


fig.write_image('../../plots/content_rating_pie.png', scale=2)
fig.show()

## Install Volume Analysis

The `Installs` column is stored as a formatted string with comma separators and a trailing `+` sign. Converting it to a numeric type is required before any install-based calculation. The distribution is heavily right-skewed: a small number of apps reach 1 billion installs while most sit below 100,000.

In [27]:
df_apps_clean.head()

,App,Category,Rating,Reviews,Size_MBs,Installs,Type,Price,Content_Rating,Genres
21,KBA-EZ Health Guide,MEDICAL,5.00,4,25.00,1,Free,0,Everyone,Medical
28,Ra Ga Ba,GAME,5.00,2,20.00,1,Paid,$1.49,Everyone,Arcade
47,Mu.F.O.,GAME,5.00,2,16.00,1,Paid,$0.99,Everyone,Arcade
82,Brick Breaker BR,GAME,5.00,7,19.00,5,Free,0,Everyone,Arcade
99,Anatomy & Physiology Vocabulary Exam Review App,MEDICAL,5.00,1,4.60,5,Free,0,Everyone,Medical


In [28]:
df_apps_clean.sort_values("Installs", ascending=False)

df_apps_clean.head(15)

,App,Category,Rating,Reviews,Size_MBs,Installs,Type,Price,Content_Rating,Genres
21,KBA-EZ Health Guide,MEDICAL,5.00,4,25.00,1,Free,0,Everyone,Medical
28,Ra Ga Ba,GAME,5.00,2,20.00,1,Paid,$1.49,Everyone,Arcade
47,Mu.F.O.,GAME,5.00,2,16.00,1,Paid,$0.99,Everyone,Arcade
82,Brick Breaker BR,GAME,5.00,7,19.00,5,Free,0,Everyone,Arcade
99,Anatomy & Physiology Vocabulary Exam Review App,MEDICAL,5.00,1,4.60,5,Free,0,Everyone,Medical
114,FK Atlantas,SPORTS,1.50,2,26.00,5,Free,0,Everyone,Sports
126,Tablet Reminder,MEDICAL,5.00,4,2.50,5,Free,0,Everyone,Medical
128,CQ ESPM,BUSINESS,5.00,2,3.40,5,Free,0,Everyone,Business
141,Clinic Doctor EHr,MEDICAL,5.00,2,7.10,5,Free,0,Everyone,Medical
151,EB Cash Collections,BUSINESS,5.00,1,4.30,5,Free,0,Everyone,Business


The conversion pipeline: strip commas → cast to numeric → group by install tier. This reveals that most apps cluster in the 1,000–100,000 install range, with only a handful reaching 1B+.

In [29]:
# checking datatype of installs column

df_apps_clean.info()


<class 'pandas.core.frame.DataFrame'>
Index: 8199 entries, 21 to 10835
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   App             8199 non-null   object 
 1   Category        8199 non-null   object 
 2   Rating          8199 non-null   float64
 3   Reviews         8199 non-null   int64  
 4   Size_MBs        8199 non-null   float64
 5   Installs        8199 non-null   object 
 6   Type            8199 non-null   object 
 7   Price           8199 non-null   object 
 8   Content_Rating  8199 non-null   object 
 9   Genres          8199 non-null   object 
dtypes: float64(2), int64(1), object(7)
memory usage: 704.6+ KB


In [30]:
# checking datatype of installs column

df_apps_clean.Installs.describe()


count          8199
unique           19
top       1,000,000
freq           1417
Name: Installs, dtype: object

#### Install Tier Distribution

In [31]:
# first try (still dtype as object with "," included)

df_apps_clean[['App','Installs']].groupby('Installs').count()

,App
Installs,
1,3
"1,000",698
"1,000,000",1417
"1,000,000,000",20
10,69
"10,000",988
"10,000,000",933
100,303
"100,000",1096


In [32]:
# now replacing the ","

df_apps_clean.Installs = (
    df_apps_clean.Installs.astype(str).str.replace(',',""))

df_apps_clean.Installs = pd.to_numeric(df_apps_clean.Installs)
df_apps_clean[['App','Installs']].groupby('Installs').count()

,App
Installs,
1,3
5,9
10,69
50,56
100,303
500,199
1000,698
5000,425
10000,988


In [33]:
df_apps_clean[df_apps_clean['Installs']==1000000000]

,App,Category,Rating,Reviews,Size_MBs,Installs,Type,Price,Content_Rating,Genres
10783,Google Play Books,BOOKS_AND_REFERENCE,3.90,1433233,5.70,1000000000,Free,0,Teen,Books & Reference
10784,Messenger – Text and Video Chat for Free,COMMUNICATION,4.00,56642847,3.50,1000000000,Free,0,Everyone,Communication
10785,WhatsApp Messenger,COMMUNICATION,4.40,69119316,3.50,1000000000,Free,0,Everyone,Communication
10786,Google Chrome: Fast & Secure,COMMUNICATION,4.30,9642995,3.50,1000000000,Free,0,Everyone,Communication
10787,Gmail,COMMUNICATION,4.30,4604324,3.50,1000000000,Free,0,Everyone,Communication
10788,Hangouts,COMMUNICATION,4.00,3419249,3.50,1000000000,Free,0,Everyone,Communication
10792,Skype - free IM & video calls,COMMUNICATION,4.10,10484169,3.50,1000000000,Free,0,Everyone,Communication
10803,Google Play Games,ENTERTAINMENT,4.30,7165362,9.35,1000000000,Free,0,Teen,Entertainment
10805,Facebook,SOCIAL,4.10,78158306,5.30,1000000000,Free,0,Teen,Social
10806,Instagram,SOCIAL,4.50,66577313,5.30,1000000000,Free,0,Teen,Social


## Revenue Estimation and Price Filtering

A ballpark revenue estimate (`Price × Installs`) requires the Price column to be numeric first — currently stored as a dollar-prefixed string. After conversion, the top 20 most expensive apps reveal a junk tier: apps listed at $200–250 with near-zero installs that exist purely as data anomalies. These are filtered out with a `Price < $250` ceiling before computing revenue estimates.

In [34]:
df_apps_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8199 entries, 21 to 10835
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   App             8199 non-null   object 
 1   Category        8199 non-null   object 
 2   Rating          8199 non-null   float64
 3   Reviews         8199 non-null   int64  
 4   Size_MBs        8199 non-null   float64
 5   Installs        8199 non-null   int64  
 6   Type            8199 non-null   object 
 7   Price           8199 non-null   object 
 8   Content_Rating  8199 non-null   object 
 9   Genres          8199 non-null   object 
dtypes: float64(2), int64(2), object(6)
memory usage: 704.6+ KB


In [35]:
df_apps_clean[["App","Price"]].groupby("Price").count()

,App
Price,
$0.99,104
$1.00,2
$1.20,1
$1.29,1
$1.49,31
...,...
$8.49,1
$8.99,4
$9.00,1


In [36]:
pd.set_option('display.max_rows', None)


In [37]:
df_apps_clean.Price = (
    df_apps_clean.Price.astype(str).str.replace("$","")
)
df_apps_clean.Price = pd.to_numeric(df_apps_clean.Price)
df_apps_clean[["App","Price"]].groupby('Price').count()


,App
Price,
0.00,7595
0.99,104
1.00,2
1.20,1
1.29,1
1.49,31
1.50,1
1.59,1
1.61,1


In [38]:
df_apps_clean[["App","Price"]].sort_values("Price",ascending=False).head(20)

,App,Price
3946,I'm Rich - Trump Edition,400.00
2461,I AM RICH PRO PLUS,399.99
4606,I Am Rich Premium,399.99
3145,I am rich(premium),399.99
3554,💎 I'm rich,399.99
5765,I am rich,399.99
1946,I am rich (Most expensive app),399.99
2775,I Am Rich Pro,399.99
3221,I am Rich Plus,399.99
3114,I am Rich,399.99


### Price Distribution: Sub-$250 Apps

After removing the junk tier, the realistic paid app market becomes visible. Most paid apps are priced below $10, with a long tail up to $250.

In [39]:
df_apps_clean = df_apps_clean[df_apps_clean["Price"]<250]
df_apps_clean.sort_values('Price', ascending=False).head(5)

,App,Category,Rating,Reviews,Size_MBs,Installs,Type,Price,Content_Rating,Genres
2281,Vargo Anesthesia Mega App,MEDICAL,4.60,92,32.00,1000,Paid,79.99,Everyone,Medical
1407,LTC AS Legal,MEDICAL,4.00,6,1.30,100,Paid,39.99,Everyone,Medical
2629,I am Rich Person,LIFESTYLE,4.20,134,1.80,1000,Paid,37.99,Everyone,Lifestyle
2481,A Manual of Acupuncture,MEDICAL,3.50,214,68.00,1000,Paid,33.99,Everyone,Medical
2463,PTA Content Master,MEDICAL,4.20,64,41.00,1000,Paid,29.99,Everyone,Medical


### Revenue Estimate: Highest Grossing Paid Apps

`Revenue Estimate = Price × Installs` — a ballpark figure that treats every install as a sale at full price. It overstates actual revenue for apps with refunds or promotional pricing, but it provides a consistent relative ranking across categories.

In [40]:
# Ensure we are working on a real DataFrame (not a slice)
df_apps_clean = df_apps_clean.copy()

# Create the Revenue_Estimate as a standalone Series
df_apps_clean["Revenue Estimate"] = (
    df_apps_clean['Installs'] * df_apps_clean['Price']
)


# Insert Revenue_Estimate as the second column (index 1)

# Show the top 10 highest-grossing apps
top_10_revenue_apps = (
    df_apps_clean
        .sort_values('Revenue Estimate', ascending=False)
        .head(10)
)

top_10_revenue_apps


,App,Category,Rating,Reviews,Size_MBs,Installs,Type,Price,Content_Rating,Genres,Revenue Estimate
9220,Minecraft,FAMILY,4.50,2376564,19.00,10000000,Paid,6.99,Everyone 10+,Arcade;Action & Adventure,"69,900,000.00"
8825,Hitman Sniper,GAME,4.60,408292,29.00,10000000,Paid,0.99,Mature 17+,Action,"9,900,000.00"
7151,Grand Theft Auto: San Andreas,GAME,4.40,348962,26.00,1000000,Paid,6.99,Mature 17+,Action,"6,990,000.00"
7477,Facetune - For Free,PHOTOGRAPHY,4.40,49553,48.00,1000000,Paid,5.99,Everyone,Photography,"5,990,000.00"
7977,Sleep as Android Unlock,LIFESTYLE,4.50,23966,0.85,1000000,Paid,5.99,Everyone,Lifestyle,"5,990,000.00"
6594,DraStic DS Emulator,GAME,4.60,87766,12.00,1000000,Paid,4.99,Everyone,Action,"4,990,000.00"
6082,Weather Live,WEATHER,4.50,76593,4.75,500000,Paid,5.99,Everyone,Weather,"2,995,000.00"
7954,Bloons TD 5,FAMILY,4.60,190086,94.00,1000000,Paid,2.99,Everyone,Strategy,"2,990,000.00"
7633,Five Nights at Freddy's,GAME,4.60,100805,50.00,1000000,Paid,2.99,Teen,Action,"2,990,000.00"
6746,Card Wars - Adventure Time,FAMILY,4.30,129603,23.00,1000000,Paid,2.99,Everyone 10+,Card;Action & Adventure,"2,990,000.00"


## Category Competition

### Number of Apps per Category

The number of competing apps per category is the supply side of the market equation. High app counts signal crowded markets where discoverability is the primary challenge.

In [41]:
top_10_category = df_apps_clean.Category.value_counts()[:10]
top_10_category

Category
FAMILY             1606
GAME                910
TOOLS               719
PRODUCTIVITY        301
PERSONALIZATION     298
LIFESTYLE           297
FINANCE             296
MEDICAL             292
PHOTOGRAPHY         263
BUSINESS            262
Name: count, dtype: int64

### Total Installs per Category

Total installs per category measures aggregate demand — but volume alone is misleading without knowing how many apps are splitting that demand. The Opportunity Score in the analysis section below combines both dimensions.

In [42]:
bar = px.bar(
    x=top_10_category.index,
    y=top_10_category.values,
    title="Top 10 App Categories by Number of Apps",)



bar.write_image('../../plots/top_10_categories_bar.png', scale=2)
bar.show()

In [43]:
pd.options.display.float_format = '{:,.0f}'.format



In [44]:
category_installs = (
    df_apps_clean
        .groupby('Category', as_index=True)
        .agg({'Installs': 'sum'})
)

category_installs.sort_values('Installs', ascending=False, inplace=True)

category_installs.head().style.format({'Installs': '{:,}'})


,Installs
Category,
GAME,"13,858,762,717"
COMMUNICATION,"11,039,241,530"
TOOLS,"8,099,724,500"
PRODUCTIVITY,"5,788,070,180"
SOCIAL,"5,487,841,475"


In [45]:
# Basic horizontal bar chart (using orientation)
h_bar = px.bar(
    x=category_installs['Installs'],
    y=category_installs.index,
    orientation='h'
)



h_bar.write_image('../../plots/category_installs_bar.png', scale=2)
h_bar.show()

In [46]:
# Horizontal bar chart with custom title and axis labels
h_bar = px.bar(
    x=category_installs['Installs'],
    y=category_installs.index,
    orientation='h',
    title='Category Popularity'
)

h_bar.update_layout(
    xaxis_title='Number of Downloads',
    yaxis_title='Category'
)



h_bar.write_image('../../plots/category_popularity_bar.png', scale=2)
h_bar.show()

### Category Concentration: Supply vs Demand

Plotting number of apps against total installs per category reveals the competitive structure of the market. Categories in the upper-left quadrant (high installs, few apps) represent the best market entry conditions. Categories in the lower-right (many apps, low installs) are structurally unattractive.

In [47]:
# df_apps_clean.info()


In [48]:
cat_number = df_apps_clean.groupby('Category').agg({'App':pd.Series.count})
cat_number

,App
Category,
ART_AND_DESIGN,61
AUTO_AND_VEHICLES,73
BEAUTY,42
BOOKS_AND_REFERENCE,169
BUSINESS,262
COMICS,54
COMMUNICATION,257
DATING,134
EDUCATION,118


#### Aggregation Patterns

In [49]:
df_apps_clean.describe()

,Rating,Reviews,Size_MBs,Installs,Price,Revenue Estimate
count,"8,184","8,184","8,184","8,184","8,184","8,184"
mean,4,"255,539",20,"9,186,103",0,"24,989"
std,1,"1,987,030",22,"58,301,374",2,"801,513"
min,1,1,0,1,0,0
25%,4,126,5,"10,000",0,0
50%,4,"3,027",11,"100,000",0,0
75%,4,"43,813",28,"1,000,000",0,0
max,5,"78,158,306",100,"1,000,000,000",80,"69,900,000"


In [50]:
df_apps_clean.head()

,App,Category,Rating,Reviews,Size_MBs,Installs,Type,Price,Content_Rating,Genres,Revenue Estimate
21,KBA-EZ Health Guide,MEDICAL,5,4,25,1,Free,0,Everyone,Medical,0
28,Ra Ga Ba,GAME,5,2,20,1,Paid,1,Everyone,Arcade,1
47,Mu.F.O.,GAME,5,2,16,1,Paid,1,Everyone,Arcade,1
82,Brick Breaker BR,GAME,5,7,19,5,Free,0,Everyone,Arcade,0
99,Anatomy & Physiology Vocabulary Exam Review App,MEDICAL,5,1,5,5,Free,0,Everyone,Medical,0


In [51]:
df_apps_clean.groupby('Category').size()

Category
ART_AND_DESIGN           61
AUTO_AND_VEHICLES        73
BEAUTY                   42
BOOKS_AND_REFERENCE     169
BUSINESS                262
COMICS                   54
COMMUNICATION           257
DATING                  134
EDUCATION               118
ENTERTAINMENT           102
EVENTS                   45
FAMILY                 1606
FINANCE                 296
FOOD_AND_DRINK           94
GAME                    910
HEALTH_AND_FITNESS      243
HOUSE_AND_HOME           62
LIBRARIES_AND_DEMO       64
LIFESTYLE               297
MAPS_AND_NAVIGATION     118
MEDICAL                 292
NEWS_AND_MAGAZINES      204
PARENTING                50
PERSONALIZATION         298
PHOTOGRAPHY             263
PRODUCTIVITY            301
SHOPPING                180
SOCIAL                  203
SPORTS                  260
TOOLS                   719
TRAVEL_AND_LOCAL        187
VIDEO_PLAYERS           148
WEATHER                  72
dtype: int64

In [52]:
df_apps_clean.groupby("Category").size()

Category
ART_AND_DESIGN           61
AUTO_AND_VEHICLES        73
BEAUTY                   42
BOOKS_AND_REFERENCE     169
BUSINESS                262
COMICS                   54
COMMUNICATION           257
DATING                  134
EDUCATION               118
ENTERTAINMENT           102
EVENTS                   45
FAMILY                 1606
FINANCE                 296
FOOD_AND_DRINK           94
GAME                    910
HEALTH_AND_FITNESS      243
HOUSE_AND_HOME           62
LIBRARIES_AND_DEMO       64
LIFESTYLE               297
MAPS_AND_NAVIGATION     118
MEDICAL                 292
NEWS_AND_MAGAZINES      204
PARENTING                50
PERSONALIZATION         298
PHOTOGRAPHY             263
PRODUCTIVITY            301
SHOPPING                180
SOCIAL                  203
SPORTS                  260
TOOLS                   719
TRAVEL_AND_LOCAL        187
VIDEO_PLAYERS           148
WEATHER                  72
dtype: int64

In [53]:
df_apps_clean.groupby("Category")["App"].count()

Category
ART_AND_DESIGN           61
AUTO_AND_VEHICLES        73
BEAUTY                   42
BOOKS_AND_REFERENCE     169
BUSINESS                262
COMICS                   54
COMMUNICATION           257
DATING                  134
EDUCATION               118
ENTERTAINMENT           102
EVENTS                   45
FAMILY                 1606
FINANCE                 296
FOOD_AND_DRINK           94
GAME                    910
HEALTH_AND_FITNESS      243
HOUSE_AND_HOME           62
LIBRARIES_AND_DEMO       64
LIFESTYLE               297
MAPS_AND_NAVIGATION     118
MEDICAL                 292
NEWS_AND_MAGAZINES      204
PARENTING                50
PERSONALIZATION         298
PHOTOGRAPHY             263
PRODUCTIVITY            301
SHOPPING                180
SOCIAL                  203
SPORTS                  260
TOOLS                   719
TRAVEL_AND_LOCAL        187
VIDEO_PLAYERS           148
WEATHER                  72
Name: App, dtype: int64

In [54]:
df_apps_clean.groupby('Category')['Installs'].sum()


Category
ART_AND_DESIGN           114233100
AUTO_AND_VEHICLES         53129800
BEAUTY                    26916200
BOOKS_AND_REFERENCE     1665791655
BUSINESS                 692018120
COMICS                    44931100
COMMUNICATION          11039241530
DATING                   140912410
EDUCATION                352852000
ENTERTAINMENT           2113660000
EVENTS                    15949410
FAMILY                  4437554490
FINANCE                  455249400
FOOD_AND_DRINK           211677750
GAME                   13858762717
HEALTH_AND_FITNESS      1134006220
HOUSE_AND_HOME            97082000
LIBRARIES_AND_DEMO        52083000
LIFESTYLE                503611120
MAPS_AND_NAVIGATION      503267560
MEDICAL                   39162676
NEWS_AND_MAGAZINES      2369110650
PARENTING                 31116110
PERSONALIZATION         1532352930
PHOTOGRAPHY             4649143130
PRODUCTIVITY            5788070180
SHOPPING                1400331540
SOCIAL                  5487841475
SPORTS     

In [55]:
df_apps_clean.groupby('Category').agg({
    'App': 'count',
    'Installs': 'sum'
})




,App,Installs
Category,,
ART_AND_DESIGN,61,114233100
AUTO_AND_VEHICLES,73,53129800
BEAUTY,42,26916200
BOOKS_AND_REFERENCE,169,1665791655
BUSINESS,262,692018120
COMICS,54,44931100
COMMUNICATION,257,11039241530
DATING,134,140912410
EDUCATION,118,352852000


In [56]:
df_apps_clean.groupby('Category').agg(
    num_apps=('App', 'count'),
    total_installs=('Installs', 'sum')
)

,num_apps,total_installs
Category,,
ART_AND_DESIGN,61,114233100
AUTO_AND_VEHICLES,73,53129800
BEAUTY,42,26916200
BOOKS_AND_REFERENCE,169,1665791655
BUSINESS,262,692018120
COMICS,54,44931100
COMMUNICATION,257,11039241530
DATING,134,140912410
EDUCATION,118,352852000


In [57]:
df_apps_clean.groupby(['Category', 'Type'])['App'].count()


Category             Type
ART_AND_DESIGN       Free      58
                     Paid       3
AUTO_AND_VEHICLES    Free      72
                     Paid       1
BEAUTY               Free      42
BOOKS_AND_REFERENCE  Free     161
                     Paid       8
BUSINESS             Free     253
                     Paid       9
COMICS               Free      54
COMMUNICATION        Free     235
                     Paid      22
DATING               Free     131
                     Paid       3
EDUCATION            Free     114
                     Paid       4
ENTERTAINMENT        Free     100
                     Paid       2
EVENTS               Free      45
FAMILY               Free    1456
                     Paid     150
FINANCE              Free     289
                     Paid       7
FOOD_AND_DRINK       Free      92
                     Paid       2
GAME                 Free     834
                     Paid      76
HEALTH_AND_FITNESS   Free     232
                     P

In [58]:
df_apps_clean.groupby('Category')['Installs'].sum().reset_index()


,Category,Installs
0,ART_AND_DESIGN,114233100
1,AUTO_AND_VEHICLES,53129800
2,BEAUTY,26916200
3,BOOKS_AND_REFERENCE,1665791655
4,BUSINESS,692018120
5,COMICS,44931100
6,COMMUNICATION,11039241530
7,DATING,140912410
8,EDUCATION,352852000
9,ENTERTAINMENT,2113660000


#### Named Aggregation Syntax

In [59]:
# df.groupby(GROUP_COLUMNS).agg(WHAT_TO_DO)


In [60]:
df_apps_clean.columns


Index(['App', 'Category', 'Rating', 'Reviews', 'Size_MBs', 'Installs', 'Type',
       'Price', 'Content_Rating', 'Genres', 'Revenue Estimate'],
      dtype='object')

In [61]:
df_apps_clean.groupby('Category').agg('count')


,App,Rating,Reviews,Size_MBs,Installs,Type,Price,Content_Rating,Genres,Revenue Estimate
Category,,,,,,,,,,
ART_AND_DESIGN,61,61,61,61,61,61,61,61,61,61
AUTO_AND_VEHICLES,73,73,73,73,73,73,73,73,73,73
BEAUTY,42,42,42,42,42,42,42,42,42,42
BOOKS_AND_REFERENCE,169,169,169,169,169,169,169,169,169,169
BUSINESS,262,262,262,262,262,262,262,262,262,262
COMICS,54,54,54,54,54,54,54,54,54,54
COMMUNICATION,257,257,257,257,257,257,257,257,257,257
DATING,134,134,134,134,134,134,134,134,134,134
EDUCATION,118,118,118,118,118,118,118,118,118,118


In [62]:
df_apps_clean.groupby('Category').agg({'App': pd.Series.count})


,App
Category,
ART_AND_DESIGN,61
AUTO_AND_VEHICLES,73
BEAUTY,42
BOOKS_AND_REFERENCE,169
BUSINESS,262
COMICS,54
COMMUNICATION,257
DATING,134
EDUCATION,118


In [63]:
df_apps_clean.groupby('Category').agg({
    'App': 'count',
    'Installs': 'sum'
})


,App,Installs
Category,,
ART_AND_DESIGN,61,114233100
AUTO_AND_VEHICLES,73,53129800
BEAUTY,42,26916200
BOOKS_AND_REFERENCE,169,1665791655
BUSINESS,262,692018120
COMICS,54,44931100
COMMUNICATION,257,11039241530
DATING,134,140912410
EDUCATION,118,352852000


In [64]:
# modern agg

df_apps_clean.groupby('Category').agg(
    num_apps=('App', 'count'),
    total_installs=('Installs', 'sum')
)


,num_apps,total_installs
Category,,
ART_AND_DESIGN,61,114233100
AUTO_AND_VEHICLES,73,53129800
BEAUTY,42,26916200
BOOKS_AND_REFERENCE,169,1665791655
BUSINESS,262,692018120
COMICS,54,44931100
COMMUNICATION,257,11039241530
DATING,134,140912410
EDUCATION,118,352852000


In [65]:
df_apps_clean.groupby('Category').agg({
    'Price': ['min', 'mean', 'max']
})


Price         
                      min mean max
Category                          
ART_AND_DESIGN          0    0   2
AUTO_AND_VEHICLES       0    0   2
BEAUTY                  0    0   0
BOOKS_AND_REFERENCE     0    0   5
BUSINESS                0    0  18
COMICS                  0    0   0
COMMUNICATION           0    0   5
DATING                  0    0   8
EDUCATION               0    0   6
ENTERTAINMENT           0    0   5
EVENTS                  0    0   0
FAMILY                  0    0  30
FINANCE                 0    0  19
FOOD_AND_DRINK          0    0   5
GAME                    0    0  18
HEALTH_AND_FITNESS      0    0   8
HOUSE_AND_HOME          0    0   0
LIBRARIES_AND_DEMO      0    0   0
LIFESTYLE               0    0  38
MAPS_AND_NAVIGATION     0    0  12
MEDICAL                 0    2  80
NEWS_AND_MAGAZINES      0    0   3
PARENTING               0    0   5
PERSONALIZATION         0    0  10
PHOTOGRAPHY             0    0  20
PRODUCTIVITY            0    0   9
SHOPPING                0    0   3
SOCIAL                  0    0   1
SPORTS                  0    0  30
TOOLS                   0    0  15
TRAVEL_AND_LOCAL        0    0   9
VIDEO_PLAYERS           0    0   6
WEATHER                 0    0   7

In [66]:
cat_number = (
    df_apps_clean
        .groupby('Category')
        .agg(
            number_apps = ("App", "count"))
)

cat_number

,number_apps
Category,
ART_AND_DESIGN,61
AUTO_AND_VEHICLES,73
BEAUTY,42
BOOKS_AND_REFERENCE,169
BUSINESS,262
COMICS,54
COMMUNICATION,257
DATING,134
EDUCATION,118


#### Category Summary Construction

In [67]:
cat_number.head()

,number_apps
Category,
ART_AND_DESIGN,61
AUTO_AND_VEHICLES,73
BEAUTY,42
BOOKS_AND_REFERENCE,169
BUSINESS,262


In [68]:
category_installs.head()

,Installs
Category,
GAME,13858762717
COMMUNICATION,11039241530
TOOLS,8099724500
PRODUCTIVITY,5788070180
SOCIAL,5487841475


In [69]:
cat_merged_df = pd.merge(
    cat_number,
    category_installs,
    on='Category',
    how='inner')
print("The dimensions of the dataframe are:", cat_merged_df.shape)

cat_merged_df.sort_values("Installs", ascending=False)

cat_merged_df.style.format({
    'Installs': '{:,}',
    'App': '{:,}'   # if App is a count column
})


The dimensions of the dataframe are: (33, 2)


,number_apps,Installs
Category,,
ART_AND_DESIGN,61,"114,233,100"
AUTO_AND_VEHICLES,73,"53,129,800"
BEAUTY,42,"26,916,200"
BOOKS_AND_REFERENCE,169,"1,665,791,655"
BUSINESS,262,"692,018,120"
COMICS,54,"44,931,100"
COMMUNICATION,257,"11,039,241,530"
DATING,134,"140,912,410"
EDUCATION,118,"352,852,000"


The scatter plot maps each category by its number of competing apps (x-axis) against total installs (y-axis), with point size encoding app count and colour encoding install volume. A log y-axis is applied to make the full range readable.

In [70]:
scatter = px.scatter(cat_merged_df,
                     x='number_apps',
                     y='Installs',
                     title="Category Concentration",
                     size="number_apps",
                     hover_name=cat_merged_df.index,
                     color='Installs',)

scatter.update_layout(
    xaxis_title='Number of Apps (Lower = More Concentrated)',
    yaxis_title='Number of Installs',
    yaxis=dict(type='log')
)


scatter.write_image('../../plots/category_concentration_scatter.png', scale=2)
scatter.show()

## Genre Analysis

The `Genres` column uses semicolon-separated values — a single app can belong to multiple genres (e.g. `Action;Action & Adventure`). Standard `.value_counts()` treats multi-genre strings as atomic values and produces incorrect counts. The correct approach splits on `;` and stacks the result to produce a flat genre series.

In [71]:
df_apps_clean.head()

,App,Category,Rating,Reviews,Size_MBs,Installs,Type,Price,Content_Rating,Genres,Revenue Estimate
21,KBA-EZ Health Guide,MEDICAL,5,4,25,1,Free,0,Everyone,Medical,0
28,Ra Ga Ba,GAME,5,2,20,1,Paid,1,Everyone,Arcade,1
47,Mu.F.O.,GAME,5,2,16,1,Paid,1,Everyone,Arcade,1
82,Brick Breaker BR,GAME,5,7,19,5,Free,0,Everyone,Arcade,0
99,Anatomy & Physiology Vocabulary Exam Review App,MEDICAL,5,1,5,5,Free,0,Everyone,Medical,0


In [72]:
genre_series = pd.Series(
    df_apps_clean['Genres'].unique(),
    name='Genre'
)

genre_series

0                                    Medical
1                                     Arcade
2                                     Sports
3                                   Business
4                          Books & Reference
5                                     Social
6                                      Tools
7                                  Education
8                              Communication
9                              Entertainment
10                              Productivity
11                                    Action
12                                Simulation
13                                    Casual
14                                 Lifestyle
15                                    Racing
16                                    Dating
17                                    Events
18                         Maps & Navigation
19                                  Shopping
20                           Personalization
21                                 Parenting
22        

In [73]:
# METHOD A: split with expand=True, then stack

stack_method = (
    df_apps_clean['Genres']
        .str.split(';', expand=True)
        .stack()
)

print(type(stack_method))
print(stack_method.head(10))
print("Shape:", stack_method.shape)


<class 'pandas.core.series.Series'>
21   0     Medical
28   0      Arcade
47   0      Arcade
82   0      Arcade
99   0     Medical
114  0      Sports
126  0     Medical
128  0    Business
141  0     Medical
151  0    Business
dtype: object
Shape: (8564,)


In [74]:
genre_counts_stack = stack_method.nunique()
genre_counts_stack


53

In [75]:
# METHOD B: split into lists, then explode

explode_method = (
    df_apps_clean['Genres']
        .str.split(';')
        .explode()
        .str.strip()
)

print(type(explode_method))
print(explode_method.head(10))
print("\n\n")
print("Shape:", explode_method.shape)

genre_counts_explode = explode_method.value_counts()
num_genres2 = len(genre_counts_explode)
print(f'Number of genres: {num_genres2}')


<class 'pandas.core.series.Series'>
21      Medical
28       Arcade
47       Arcade
82       Arcade
99      Medical
114      Sports
126     Medical
128    Business
141     Medical
151    Business
Name: Genres, dtype: object



Shape: (8564,)
Number of genres: 53


In [76]:
# Work on a safe copy
df_apps_clean = df_apps_clean.copy()

# Split Genres into at most 2 columns
genres_split = df_apps_clean['Genres'].str.split(';', n=1, expand=True)

# Assign clear column names
df_apps_clean['Genre_1'] = genres_split[0]
df_apps_clean['Genre_2'] = genres_split[1]

# Optional cleanup
df_apps_clean['Genre_1'] = df_apps_clean['Genre_1'].str.strip()
df_apps_clean['Genre_2'] = df_apps_clean['Genre_2'].str.strip()

df_apps_clean[['Genres', 'Genre_1', 'Genre_2']].head()


,Genres,Genre_1,Genre_2
21,Medical,Medical,None
28,Arcade,Arcade,None
47,Arcade,Arcade,None
82,Arcade,Arcade,None
99,Medical,Medical,None


In [77]:
# Split the strings on the semi-colon and then .stack them.
stack = df_apps_clean.Genres.str.split(';', expand=True).stack()
print(f'We now have a single column with shape: {stack.shape}')
num_genres = stack.value_counts()
print(f'Number of genres: {len(num_genres)}')


We now have a single column with shape: (8564,)
Number of genres: 53


## Genre Competition

With genres properly unpacked, the frequency distribution shows which genres dominate the store. Colour intensity encodes count, making the long tail visible at a glance.

In [78]:
print(num_genres)

Tools                      719
Education                  587
Entertainment              498
Action                     304
Productivity               301
Personalization            298
Lifestyle                  298
Finance                    296
Medical                    292
Sports                     270
Photography                263
Business                   262
Communication              258
Health & Fitness           245
Casual                     216
News & Magazines           204
Social                     203
Simulation                 200
Travel & Local             187
Arcade                     185
Shopping                   180
Books & Reference          171
Video Players & Editors    150
Dating                     134
Puzzle                     124
Maps & Navigation          118
Role Playing               111
Racing                     103
Action & Adventure          96
Strategy                    95
Food & Drink                94
Educational                 93
Adventur

The top 15 genres are plotted with a continuous colour scale (Agsunset) encoding frequency. The colour axis is hidden to keep the chart clean — the bar length already encodes the same information.

In [79]:
bar = px.bar(x = num_genres.index[:15], # index = category name
            y = num_genres.values[:15], # count
            title='Top Genres',
            hover_name=num_genres.index[:15],
            color=num_genres.values[:15],
            color_continuous_scale='Agsunset')

bar.update_layout(xaxis_title='Genre',
yaxis_title='Number of Apps',
coloraxis_showscale=False)



bar.write_image('../../plots/top_genres_bar.png', scale=2)
bar.show()

## Free vs Paid App Split by Category

The ratio of free to paid apps varies significantly by category and reveals where developers have converged on monetisation strategy. Categories with very few paid apps signal that users in that space have a strong preference for free.

In [80]:
df_free_vs_paid = df_apps_clean.groupby(["Category", "Type"], as_index=False).agg({'App': pd.Series.count})
df_free_vs_paid.head()

,Category,Type,App
0,ART_AND_DESIGN,Free,58
1,ART_AND_DESIGN,Paid,3
2,AUTO_AND_VEHICLES,Free,72
3,AUTO_AND_VEHICLES,Paid,1
4,BEAUTY,Free,42


In [81]:
df_free_vs_paid2 = df_apps_clean.groupby(["Category", "Type"], as_index=True).agg({'App': pd.Series.count})
df_free_vs_paid2.head()

App
Category          Type     
ART_AND_DESIGN    Free   58
                  Paid    3
AUTO_AND_VEHICLES Free   72
                  Paid    1
BEAUTY            Free   42

A grouped bar chart — one bar per type per category — ordered by total app count descending. The free/paid ratio is immediately visible for each category.

In [82]:
g_bar = px.bar(df_free_vs_paid,
                x='Category',
                y='App',
                title='Free vs Paid Apps by Category',
                color='Type',
                barmode='group')
    
g_bar.update_layout(xaxis_title='Category',
                    yaxis_title='Number of Apps',
                    xaxis={'categoryorder':'total descending'},
                    yaxis=dict(type='log'))
    


g_bar.write_image('../../plots/free_vs_paid_bar.png', scale=2)
g_bar.show()

## Download Cost of Charging: Free vs Paid Install Distribution

Choosing a paid price point has a direct cost: fewer downloads. This box plot quantifies the median install gap between free and paid apps, making the trade-off concrete before any revenue estimate is applied.

In [83]:
df_apps_clean.head()

,App,Category,Rating,Reviews,Size_MBs,Installs,Type,Price,Content_Rating,Genres,Revenue Estimate,Genre_1,Genre_2
21,KBA-EZ Health Guide,MEDICAL,5,4,25,1,Free,0,Everyone,Medical,0,Medical,None
28,Ra Ga Ba,GAME,5,2,20,1,Paid,1,Everyone,Arcade,1,Arcade,None
47,Mu.F.O.,GAME,5,2,16,1,Paid,1,Everyone,Arcade,1,Arcade,None
82,Brick Breaker BR,GAME,5,7,19,5,Free,0,Everyone,Arcade,0,Arcade,None
99,Anatomy & Physiology Vocabulary Exam Review App,MEDICAL,5,1,5,5,Free,0,Everyone,Medical,0,Medical,None


In [84]:
box = px.box(df_apps_clean,
                y='Installs',
                x='Type',
                color='Type',
                notched=True,
                points='all',
                title='How Many Downloads are Paid Apps Giving Up?')
    
box.update_layout(yaxis=dict(type='log'))
    


box.write_image('../../plots/installs_free_vs_paid_box.png', scale=2)
box.show()

## Revenue Estimate by Category

Box plots of `Price × Installs` per category for paid apps only. The distribution width reveals how variable revenue is within each category — a wide box means a small number of outlier apps are capturing most of the revenue.

In [85]:
df_paid_apps = df_apps_clean[df_apps_clean['Type'] == 'Paid']

box = px.box(df_paid_apps, 
                x='Category', 
                y='Revenue Estimate',
                title='How Much Can Paid Apps Earn?')
    
box.update_layout(xaxis_title='Category',
                    yaxis_title='Paid App Ballpark Revenue',
                    xaxis={'categoryorder':'min ascending'},
                    yaxis=dict(type='log'))
    
    


box.write_image('../../plots/revenue_by_category_box.png', scale=2)
box.show()

## Paid App Pricing Strategies by Category

Median paid app price varies from under $1 to over $10 depending on category. Categories with higher median prices signal audiences that treat apps as professional tools rather than casual entertainment — a key input into the viability analysis below.

In [86]:
median_price_app = df_paid_apps.Price.median()
median_price_app

2.99

In [ ]:
box = px.box(df_paid_apps,
                x='Category',
                y="Price",
                title='Price per Category')
    
box.update_layout(xaxis_title='Category',
                    yaxis_title='Paid App Price',
                    xaxis={'categoryorder':'max descending'},
                    yaxis=dict(type='log'))
    


box.write_image('../../plots/price_by_category_box.png', scale=2)
box.show()

---

## Opportunity Score by Category

**Business question:** Which categories have enough demand relative to competition to make entry worthwhile — and which are so crowded that each app captures negligible share?

**Method:** Opportunity Score = total installs ÷ number of competing apps per category. A high score means concentrated demand with few competitors. A low score means the available installs are split across many apps with little left per entrant. Categories are classified as **Opportunity** (top quartile), **Competitive**, or **Saturated** (bottom quartile) using p75 and p25 thresholds.

In [ ]:
from pathlib import Path
Path('../../plots').mkdir(parents=True, exist_ok=True)

import numpy as np
import plotly.express as px

cat_summary = df_apps_clean.groupby('Category').agg(
    num_apps=('App', 'count'),
    total_installs=('Installs', 'sum')
).reset_index()

cat_summary['opportunity_score'] = cat_summary['total_installs'] / cat_summary['num_apps']
cat_summary.sort_values('opportunity_score', ascending=False, inplace=True)

p75 = cat_summary['opportunity_score'].quantile(0.75)
p25 = cat_summary['opportunity_score'].quantile(0.25)

cat_summary['market_status'] = cat_summary['opportunity_score'].apply(
    lambda s: 'Opportunity' if s >= p75 else ('Saturated' if s <= p25 else 'Competitive')
)

cat_summary.head()

In [ ]:
opp_bar = px.bar(
    cat_summary,
    x='opportunity_score',
    y='Category',
    orientation='h',
    color='opportunity_score',
    color_continuous_scale='RdYlGn',
    title='Opportunity Score by Category (Installs per App)'
)
opp_bar.update_layout(
    xaxis_title='Opportunity Score (Total Installs / Number of Apps)',
    yaxis_title='Category',
    yaxis=dict(autorange='reversed'),
    height=700
)
opp_bar.write_image('../../plots/opportunity_score_by_category.png', scale=2)
opp_bar.show()

In [ ]:
top3_opp = cat_summary.nlargest(3, 'opportunity_score')[['Category','opportunity_score','market_status']]
top3_sat = cat_summary.nsmallest(3, 'opportunity_score')[['Category','opportunity_score','market_status']]
print('Top 3 Opportunity Categories:')
print(top3_opp.to_string(index=False))
print()
print('Top 3 Saturated Categories:')
print(top3_sat.to_string(index=False))

**Finding:** Communication leads with an Opportunity Score of **42,954,247** — 42.9 million installs per competing app. Social (27.0M) and Video Players (26.5M) round out the top three. At the other end, Medical scores just 134,119 installs per app, making Communication **320× more attractive per entrant** than the most saturated category. Events (354K) and Parenting (622K) complete the bottom three. The gap between Opportunity and Saturated categories is not marginal — it is structural.

---

## Paid App Viability by Category

**Business question:** In which categories do paid apps generate revenue meaningful enough to justify a price tag — and where is free so dominant that charging is commercially irrational?

**Method:** Filter to paid apps priced at $29.99 or below (a principled ceiling that removes data-entry junk at $200+ without discarding legitimate professional apps). Require a minimum of 10 paid apps per category to ensure the median reflects a real market, not a single outlier. Rank by median revenue estimate.

In [ ]:
df_paid_viable = df_apps_clean[
    (df_apps_clean['Type'] == 'Paid') &
    (df_apps_clean['Price'] <= 29.99)
].copy()

paid_cat = df_paid_viable.groupby('Category').agg(
    median_revenue=('Revenue Estimate', 'median'),
    median_price=('Price', 'median'),
    paid_app_count=('App', 'count')
).reset_index()

paid_cat = paid_cat[paid_cat['paid_app_count'] >= 10]
paid_cat.sort_values('median_revenue', ascending=False, inplace=True)

print(f'Categories with ≥10 paid apps (price ≤ $29.99): {len(paid_cat)}')
paid_cat

In [ ]:
paid_scatter = px.scatter(
    paid_cat,
    x='median_price',
    y='median_revenue',
    size='paid_app_count',
    text='Category',
    title='Paid App Viability: Median Price vs Median Revenue Estimate',
    color='median_revenue',
    color_continuous_scale='Blues'
)
paid_scatter.update_traces(textposition='top center')
paid_scatter.update_layout(
    xaxis_title='Median Price ($)',
    yaxis_title='Median Revenue Estimate ($)',
    height=600
)
paid_scatter.write_image('../../plots/paid_app_viability_scatter.png', scale=2)
paid_scatter.show()

**Finding:** Only **11 categories** meet the viability threshold (≥10 paid apps, price ≤ $29.99). Health & Fitness leads with a median revenue estimate of **$99,500**, followed by Games ($64,900) and Family ($24,950). Medical, despite its saturated Opportunity Score, ranks fourth at $14,970 — confirming that paid pricing can work in niche professional categories even where competition is dense. The majority of the store's categories do not appear in this table at all: they lack the paid app volume to produce a credible estimate, which is itself a signal that free-with-ads is the market consensus there.

---

## K-Means App Segmentation with PCA

**Business question:** What distinct market positions exist across 8,199 individual apps — and which segment should a new developer target?

**Method:** KMeans (k=4) applied to a StandardScaler-normalised matrix of five features: `log(Installs+1)`, `Rating`, `log(Reviews+1)`, `Price`, and `Size_MBs`. Log transforms on installs and reviews correct severe right-skew before normalisation — without them, install magnitude dominates all other features and the clusters collapse. PCA (n=2) is applied post-clustering to visualise segment separation, with explained variance displayed on each axis.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
features = ['Installs', 'Rating', 'Reviews', 'Price', 'Size_MBs']
df_seg = df_apps_clean[features].dropna().copy()

df_seg['log_installs'] = np.log1p(df_seg['Installs'])
df_seg['log_reviews'] = np.log1p(df_seg['Reviews'])

X = df_seg[['log_installs', 'Rating', 'log_reviews', 'Price', 'Size_MBs']]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_seg['Segment'] = kmeans.fit_predict(X_scaled)

print('Apps per segment:')
print(df_seg['Segment'].value_counts().sort_index())

In [ ]:
# Inspect centroids in original (human-readable) scale
centroid_df = df_seg.groupby('Segment')[['log_installs','Rating','log_reviews','Price','Size_MBs']].mean()
centroid_df['approx_installs'] = np.expm1(centroid_df['log_installs']).round(0)
centroid_df['approx_reviews'] = np.expm1(centroid_df['log_reviews']).round(0)
print('Cluster centroids (mean values per segment):')
centroid_df

In [ ]:
# Assign business labels based on actual centroid values
segment_labels = {}
for seg_id in sorted(df_seg['Segment'].unique()):
    c = centroid_df.loc[seg_id]
    if c['Price'] > 1.0:
        segment_labels[seg_id] = 'Premium Paid'
    elif c['Rating'] >= 4.2 and c['approx_reviews'] < 5000:
        segment_labels[seg_id] = 'Hidden Gems'
    elif c['approx_installs'] >= 500000:
        segment_labels[seg_id] = 'Mass Market Free'
    else:
        segment_labels[seg_id] = 'Struggling'

df_seg['Segment_Label'] = df_seg['Segment'].map(segment_labels)
print('Segment label mapping:', segment_labels)
print()
print(df_seg['Segment_Label'].value_counts())

In [ ]:
# PCA visualisation
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

var1 = pca.explained_variance_ratio_[0] * 100
var2 = pca.explained_variance_ratio_[1] * 100

df_plot = df_seg.copy()
df_plot['PC1'] = X_pca[:, 0]
df_plot['PC2'] = X_pca[:, 1]

# Centroids in PCA space
centroids_pca = pca.transform(scaler.transform(
    centroid_df[['log_installs','Rating','log_reviews','Price','Size_MBs']].values
))

color_map = {
    'Mass Market Free': '#2196F3',
    'Premium Paid': '#FF9800',
    'Hidden Gems': '#4CAF50',
    'Struggling': '#F44336'
}

fig_pca = px.scatter(
    df_plot,
    x='PC1', y='PC2',
    color='Segment_Label',
    color_discrete_map=color_map,
    opacity=0.5,
    title='App Market Segments — PCA Projection (K-Means k=4)',
    labels={
        'PC1': f'PC1 ({var1:.1f}% variance explained)',
        'PC2': f'PC2 ({var2:.1f}% variance explained)'
    }
)

# Add centroid markers
for i, (label, _) in enumerate(segment_labels.items()):
    name = segment_labels[label]
    fig_pca.add_scatter(
        x=[centroids_pca[label, 0]],
        y=[centroids_pca[label, 1]],
        mode='markers',
        marker=dict(symbol='star', size=18,
                    color=color_map.get(name, 'black'), line=dict(width=2, color='black')),
        name=f'{name} (centroid)',
        showlegend=True
    )

fig_pca.update_layout(height=600)
fig_pca.write_image('../../plots/app_segments_pca.png', scale=2)
fig_pca.show()

In [ ]:
# Segment profile table — mean unscaled features + app count
profile = df_seg.groupby('Segment_Label').agg(
    app_count=('Segment', 'count'),
    mean_installs=('Installs', 'mean'),
    mean_rating=('Rating', 'mean'),
    mean_reviews=('Reviews', 'mean'),
    mean_price=('Price', 'mean'),
    mean_size_mb=('Size_MBs', 'mean')
).round(2)
profile['mean_installs'] = profile['mean_installs'].apply(lambda x: f'{x:,.0f}')
profile['mean_reviews'] = profile['mean_reviews'].apply(lambda x: f'{x:,.0f}')
print('Segment Profile Table:')
profile

In [ ]:
# Top 3 categories per segment by app count
df_seg_cats = df_apps_clean.loc[df_seg.index].copy()
df_seg_cats['Segment_Label'] = df_seg['Segment_Label']

for seg in sorted(df_seg_cats['Segment_Label'].unique()):
    top_cats = (
        df_seg_cats[df_seg_cats['Segment_Label'] == seg]
        .groupby('Category')['App'].count()
        .nlargest(3)
    )
    print(f'--- {seg} ---')
    print(top_cats.to_string())
    print()

**Findings:**

- **Mass Market Free** is the dominant segment with **4,646 apps** and a mean install count of **14.8 million** — the highest of any cluster. FAMILY (822 apps), GAME (734), and TOOLS (340) lead this segment by app count. Zero price friction and broad category coverage make this the default competitive environment for most apps.

- **Struggling** contains **3,476 apps** with a mean of just 29,000 installs — roughly 500× fewer than Mass Market Free. The category distribution mirrors the market at large, suggesting these are not niche apps but mainstream apps that have not found traction.

- **Premium Paid** is a small, structurally isolated cluster of **62 apps** with a mean price of $17 and 23,000 mean installs. Medical (23 apps) dominates, confirming that paid pricing at scale requires professional utility or regulatory context.

- **For a new developer**, Mass Market Free is the recommended entry segment: highest install ceiling, broadest distribution channels, and no price friction barrier. Premium Paid demands an existing reputation or a category where professional audiences have a proven willingness to pay — Medical being the clearest example.

- **PCA explains 42.1% of variance on PC1** (install and review volume) and **20.9% on PC2** (price and rating). All three segments are visually separable in the two-component projection, validating the k=4 structure.

---

## Summary of Findings

**Dataset:** 8,199 Android apps after cleaning 10,841 raw records — removing NaN ratings, deduplicating on `[App, Type, Price]`, and filtering junk-priced entries above $250.

**Content distribution:** Over 80% of apps are rated Everyone, confirming the Play Store is built primarily for general audiences. Teen and Mature 17+ together account for under 15%.

**Category competition:** FAMILY and GAME have the most competing apps. COMMUNICATION leads total install volume with over 11 billion aggregate installs across 257 apps.

**Opportunity Score:** Communication scores 42.9M installs per competing app — 320× higher than Medical (134K), the most saturated category. Social and Video Players round out the top three. The score provides a directly actionable market entry signal that raw install volume alone cannot.

**Paid viability:** 11 categories sustain paid apps at credible revenue levels after filtering to ≤$29.99 with ≥10 paid apps. Health & Fitness leads at $99,500 median revenue estimate; Games ($64,900) and Family ($24,950) follow. Most categories do not qualify — free is the market consensus outside professional niches.

**K-Means segmentation:** Three distinct market positions emerge from clustering 8,199 apps on five normalised features. Mass Market Free (4,646 apps, 14.8M mean installs) dominates. Struggling (3,476 apps, 29K installs) represents the long tail of apps that have not found distribution. Premium Paid (62 apps, $17 mean price) occupies a structurally separate position driven by professional categories.

**PCA:** PC1 (42.1% variance) captures install and review volume. PC2 (20.9%) captures price and rating. The three segments are visually separable in two-component space — the k=4 structure is geometrically sound.